In [2]:
##user inputs:
#source data: go to the notebook cell headed with "##create worbook for output, test saving a copy"
#selects: go to the notebook cell headed with "##make selects so that they can easily be read in the notebook"; specify input file, input sheets, and output file
#packages: if you don't have them, install the packages under "## packages, run once globally"


##TODO: Allow for other trimming methods beyond ex hilo, e.g. rank-based?
##TODO: Add exposure?
##TODO: Check (III) and (IV) below if we have a non-square triangle (i.e. more AYs than age columns)
##TODO: Update (III) and (IV) below if we have a non-square triangle (i.e. missing early evaluations for older AYs)
##TODO: Selects in Shiny or DataEditR? I didn't use this specifically so that my selects can be followed by reading the notebook. So the selects and notes are hardcoded.
##TODO: plots/markdown?

##Approach: I think the more standard approach is to use dataframes for everything and then look at them in pivoted representation. I want to try keeping things in matrix form.

In [3]:
## packages, run once globally
# install.packages("openxlsx2")
# install.packages("lubridate")
# install.packages("tidyverse")
# install.packages("zoo")

## didn't use
# install.packages("readxl")
# install.packages("openxlsx")
# install.packages("cellranger")
# install.packages("Matrix")
# install.packages("DataEditR")
# install.packages("DT")
# install.packages("shiny")

In [22]:
##create worbook for output, test saving a copy
library(openxlsx2)
input_path = "sourcedata/PYRQ.xlsx"
output_path = "output/PYRQ_output.xlsx"

wb <- wb_load(input)
wb_save(wb,output,overwrite = TRUE)

#create styles
dec3="0.000"
dec0="#,##0"
decY="###0"

#function to create a sheet destructively; can use this to clear a sheet also
newsheet_destructive <- function(name) {
    if (name %in% wb$sheet_names) wb$remove_worksheet(sheet=name)
    wb$add_worksheet(name)
}

#function to populate a sheet
popsheet <- function(name,frame,title,style,titlerow,startcolumn) {
    wb$add_data(sheet = name, title, start_row = titlerow, start_col = startcolumn, row_names=FALSE)
    wb$add_data(sheet = name, frame, start_row = titlerow+2, start_col = startcolumn, row_names=TRUE)
    wb$add_numfmt(sheet=name, numfmt=style, 
         dims=wb_dims(rows=(titlerow+3):(nrow(frame)+(titlerow+3)), cols=(startcolumn+1):(ncol(frame)+startcolumn+1)))
    wb$add_font(sheet=name, bold=TRUE,
        dims=wb_dims(rows=(titlerow+3):(nrow(frame)+(titlerow+3)),cols=startcolumn:startcolumn))
    wb$add_font(sheet=name, bold=TRUE,
        dims=wb_dims(rows=(titlerow):(titlerow+2),cols=startcolumn:(ncol(frame)+startcolumn+1)))
}

#function to get max row and column
bounds <- function(name) {
  ws <- wb$worksheets[[which(wb$sheet_names == name)]]
  cc <- ws$sheet_data$cc
  max_row <- max(as.integer(cc$row_r))
  max_col <- max(col2int(cc$c_r))
  return(c(max_row, max_col))
}

#function to set column widths
autowidth <- function(name,col=-1,w=-1) {
    if (col==-1) ncols=bounds(name)[2] else ncols=col
    if (w == -1) width="auto" else width=w
    wb$set_col_widths(name, cols=1:ncols, widths=width)
}

In [5]:
##look at list of worksheets
sheets <- wb_get_sheet_names(wb)
data.frame(index = seq_along(sheets), name = sheets)


,index,name
,<int>,<chr>
ex1_curr,1,ex1_curr
ex1_prior,2,ex1_prior
ex2_tri,3,ex2_tri


In [6]:
##read the triangle, create matrix, create variables to be used for checks

library(lubridate)
library(tidyverse)

#I don't want scientific notation in output
options(scipen=999)

#load the triangle
m <- as.matrix(df <- wb_to_df(wb, sheet="ex2_tri"))

#make it a matrix
#I will be switching back and forth between "matrix form" and "tall skinny form" to make different calculations easier
m <- m[, colSums(is.na(m)) != nrow(m)]
m <- m[rowSums(is.na(m)) != ncol(m),]

#label the rows
rownames(m) = m[,"acc_yr"]

#get min AY for checks
AY_min=min(m[,1])
AY_max=max(m[,1])

# remove labels from the body of the matrxi now that they are labels proper
m = m[,-1]

#get age characteristics for checks
age_min <- as.integer(colnames(m)[1])
age_step <- as.integer(colnames(m)[2])-as.integer(colnames(m)[1])
age_count <- ncol(m)
age_max <- as.integer(colnames(m)[age_count])



Attaching package: ‘lubridate’


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr   1.2.1     ✔ readr   2.2.0
✔ forcats 1.0.1     ✔ stringr 1.6.0
✔ ggplot2 4.0.3     ✔ tibble  3.3.1
✔ purrr   1.2.2     ✔ tidyr   1.3.2
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [ ]:
## make the matrix back into a dataframe to do more checks; remove NA now that we understand the pattern of NAs; create DevDate column
df <- as.data.frame(as.table(m))
names(df) <- c("AY", "Age", "Loss")
df <- df[!is.na(df[,"Loss"]),]
df[,"AY"] <- as.numeric(as.character(df[,"AY"]))
df[,"Age"] <- as.numeric(as.character(df[,"Age"]))
df[, "DevDate"] <- as.Date(-1+ISOdate(df[,"AY"]+1,1,1))
df[, "DevDate"] <- df[, "DevDate"]-months(12)+months(df[,"Age"])-days(1)


#(I) the filter 3 lines down should do nothing; this will be validated when I compare m2 and m below 
DevDate_max <- max(df[,"DevDate"])
#print(nrow(df))
df <- df[(df[,"DevDate"]<=DevDate_max),]
#print(nrow(df))

#turn this back into a matrix
m2<-as.matrix(pivot_wider(df,id_cols="AY",names_from="Age", values_from="Loss"))
m2<- m2[,-1]

#checks
triangle_shape_summary <- data.frame(key=character(), value=numeric())
test_errors <- data.frame(message=character(), error=numeric())
triangle_shape_summary=rbind(triangle_shape_summary,data.frame(key="min AY",value=AY_min))
triangle_shape_summary=rbind(triangle_shape_summary,data.frame(key="max AY",value=AY_max))
triangle_shape_summary=rbind(triangle_shape_summary,data.frame(key="min age",value=age_min))
triangle_shape_summary=rbind(triangle_shape_summary,data.frame(key="age step",value=age_step))
triangle_shape_summary=rbind(triangle_shape_summary,data.frame(key="max age",value=age_max))
triangle_shape_summary=rbind(triangle_shape_summary,data.frame(key="age count",value=age_count))

#function to elementwise compare matrix elements without freaking out over NA; this version will count NA=NaN
equal <- function(x, y) {(is.na(x) == is.na(y)) & (is.na(x) | x==y)}

#(II) this next line checks that I did the matrix-> df conversion correctly and also that the DevDate filter above under (I) did nothing
if(all(equal(m2,m))) {message="matrices are equal";error=0} else {message="matrices are unequal";error=1}
    test_errors=rbind(test_errors,data.frame(message=message,error=error))

#(III) this next line does not by itself check that the PATTERN of NAs is correct; in theory, each row could have the right number of NAs but in the wrong place. But together with the check (II) above and the lack of tidyverse warnings when converting df to m2, it does work
#this next would need more checking if we didn't have a "square" triangle; it is supposed to continue working if we get more accident years but not more maturities
if (all(pmin(ncol(m),nrow(m)-1:nrow(m)+1)==nrow(m)-rowSums(is.na(m)))) {message="NAs are in triangle pattern";error=0} else {message="CHECK that this triangle is in 'upper triangular' format with NA in the bottom right half";error=1}
    test_errors=rbind(test_errors,data.frame(message=message,error=error))
if (all(1:nrow(m)+AY_min-1==as.numeric(rownames(m)))) {message="AYs as expected";error=0} else {message="CHECK that AYs are consecutive, annual, numerical, and not duplicated";error=1}
    test_errors=rbind(test_errors,data.frame(message=message,error=error))
if (all(as.numeric(colnames(m))==(1:ncol(m)-1)*age_step+age_min)) {message="ages are in linear pattern";error=0} else {message="CHECK that ages are in linear pattern";error=1}
    test_errors=rbind(test_errors,data.frame(message=message,error=error))

#print errors
test_errors
print(paste0("error count: ",sum(test_errors[,"error"])))

#write errors and triangle summary
newsheet_destructive("initial diagnostics")
popsheet("initial diagnostics",triangle_shape_summary,"triangle shape summary",decY,1,1)
popsheet("initial diagnostics",test_errors,"initial errors",decY,1,4)
autowidth("initial diagnostics")
wb_save(wb, output_path, overwrite = TRUE)


message,error
<chr>,<dbl>
matrices are equal,0
NAs are in triangle pattern,0
AYs as expected,0
ages are in linear pattern,0


[1] "error count: 0"


In [8]:
##make LDF matrices
library(zoo)

#normal LDF matrix
l_avg1 <- m[,-1] / m[,-ncol(m)]
print("LDFs:")
l_avg1

#ex hilo:
count <- function(x) sum(!is.na(x))
rollcount<- function(x) rollapply(x, width=nrow(m), FUN=count, partial=TRUE, align="right")
rollsum<- function(x) rollapply(x, width=nrow(m), FUN=sum, partial=TRUE, align="right")
rollmin<- function(x) rollapply(x, width=nrow(m), FUN=min, partial=TRUE, align="right")
rollmax<- function(x) rollapply(x, width=nrow(m), FUN=max, partial=TRUE, align="right")
l_avgExH1L1 <- (apply(l_avg1,2,rollsum) - apply(l_avg1,2,rollmin) - apply(l_avg1,2,rollmax))/(apply(l_avg1,2,rollcount)-2)
l_avgExH1L1[1:2,] <- NA
rownames(l_avgExH1L1) <- rownames(m)
print("avg ex Hi Lo:")
l_avgExH1L1

#wtd(4) where 4 points are available
i = 4
roll<- function(x) rollapply(x, width=i, FUN=sum, partial=FALSE, align="right")
l_wtd4 <- apply(m[,-1],2,roll) / apply(m[,-ncol(m)],2,roll)
l_wtd4 <- rbind(matrix(NA,nrow=i-1,ncol=ncol(l_wtd4)),l_wtd4)
rownames(l_wtd4) <- rownames(m)
print(paste0("Wtd ",i,":"))
l_wtd4

#wtd(2) where 2 points are available
i = 2
roll<- function(x) rollapply(x, width=i, FUN=sum, partial=FALSE, align="right")
l_wtd2 <- apply(m[,-1],2,roll) / apply(m[,-ncol(m)],2,roll)
l_wtd2 <- rbind(matrix(NA,nrow=i-1,ncol=ncol(l_wtd2)),l_wtd2)
rownames(l_wtd2) <- rownames(m)
print(paste0("Wtd ",i,":"))
l_wtd2

#wtd(all)
roll<- function(x) rollapply(x, width=nrow(m), FUN=sum, partial=TRUE, align="right")
l_wtdAll <- apply(m[,-1],2,roll) / apply(m[,-ncol(m)],2,roll)
rownames(l_wtdAll) <- rownames(m)
print("Wtd All:")
l_wtdAll


Attaching package: ‘zoo’


The following objects are masked from ‘package:base’:

    as.Date, as.Date.numeric




[1] "LDFs:"


,24,36,48,60,72,84,96
2015,1.492754,1.123786,0.9395248,1.011494,1.011364,1.051685,1.004274
2016,1.608150,1.111111,1.5052632,1.001166,1.046566,1.027809,NA
2017,1.304833,1.049858,1.0379919,1.241830,1.058947,NA,NA
2018,1.496094,1.070496,0.9658537,1.010101,NA,NA,NA
2019,1.980344,1.260546,1.0039370,NA,NA,NA,NA
2020,1.523697,1.200622,NA,NA,NA,NA,NA
2021,1.802857,NA,NA,NA,NA,NA,NA
2022,NA,NA,NA,NA,NA,NA,NA


[1] "avg ex Hi Lo:"


,24,36,48,60,72,84,96
2015,NA,NA,NA,NA,NA,NA,NA
2016,NA,NA,NA,NA,NA,NA,NA
2017,1.492754,1.111111,1.037992,1.011494,1.046566,NA,NA
2018,1.494424,1.090804,1.001923,1.010798,NA,NA,NA
2019,1.532333,1.101798,1.002594,NA,NA,NA,NA
2020,1.530174,1.126504,NA,NA,NA,NA,NA
2021,1.584710,NA,NA,NA,NA,NA,NA
2022,NA,NA,NA,NA,NA,NA,NA


...

,24,36,48,60,72,84,96
2015,NA,NA,NA,NA,NA,NA,NA
2016,NA,NA,NA,NA,NA,NA,NA
2017,NA,NA,NA,NA,NA,NA,NA
2018,1.447084,1.084577,1.125688,1.079462,NA,NA,NA
2019,1.581579,1.136855,1.111965,NA,NA,NA,NA
2020,1.561306,1.158248,NA,NA,NA,NA,NA
2021,1.716376,NA,NA,NA,NA,NA,NA
2022,NA,NA,NA,NA,NA,NA,NA


[1] "Wtd 2:"


,24,36,48,60,72,84,96
2015,NA,NA,NA,NA,NA,NA,NA
2016,1.554622,1.116757,1.2516941,1.004640,1.034642,1.035714,NA
2017,1.417736,1.075720,1.2417751,1.114603,1.053068,NA,NA
2018,1.366499,1.057143,1.0122058,1.162791,NA,NA,NA
2019,1.793363,1.199327,0.9929874,NA,NA,NA,NA
2020,1.747889,1.233954,NA,NA,NA,NA,NA
2021,1.650259,NA,NA,NA,NA,NA,NA
2022,NA,NA,NA,NA,NA,NA,NA


[1] "Wtd All:"


,24,36,48,60,72,84,96
2015,1.492754,1.123786,0.9395248,1.011494,1.011364,1.051685,1.004274
2016,1.554622,1.116757,1.2516941,1.004640,1.034642,1.035714,NA
2017,1.436011,1.087892,1.1627119,1.092809,1.044909,NA,NA
2018,1.447084,1.084577,1.1256881,1.079462,NA,NA,NA
2019,1.567929,1.134943,1.0869837,NA,NA,NA,NA
2020,1.559513,1.147152,NA,NA,NA,NA,NA
2021,1.592679,NA,NA,NA,NA,NA,NA
2022,NA,NA,NA,NA,NA,NA,NA


In [9]:
##make tables of standard methods and current evaluation, get ready to make selects

#function to get the nth-latest diagonal starting from 0:
dg <- function(x,o) {
    if (o<0) diag(x[nrow(x):1, ][o,])
    else if (o==0) diag(x[nrow(x):1, ])
    else if (o>0) diag(x[nrow(x):1, ][,-o])
}

#get the range of standard methods as a row
Methods <- rbind(dg(l_avgExH1L1,-1),dg(l_wtd4,-1),dg(l_wtd2,-1),dg(l_wtdAll,-1))
rownames(Methods) <- c("Avg ex HiLo","Wtd 4","Wtd 2", "Wtd All")
colnames(Methods) <- colnames(l_avg1)

#(IV) this would need to be updated for non-square triangles
#get the current evaluations as a row, with AY and age, and a space for projected at the bottom
Current <- {rbind(
    as.numeric(rownames(m)[nrow(m):1]),
    dg(m,0),
    matrix(as.numeric(NA),1,ncol(m)))
}

#create a matrix for the selections, including tail
Sel <- rbind(matrix(as.numeric(NA),2,ncol(Methods)+1))

#create a frame for the selection notes, including tail
SelNotes <- data.frame(matrix("", 1, ncol(Methods)+1))

rownames(Current) <- c("AY","Current","ProjUlt")
colnames(Current) <- colnames(m)
rownames(Sel) = c("Sel Inc","Sel Cum")
colnames(Sel) <- c(colnames(l_avg1), "tail")
rownames(SelNotes) <- c("Notes")
colnames(SelNotes) <- colnames(Sel)

#print outputs to check for reasonableness
#in the future, maybe these triangles could be used for backtesting
print("historical triangle:")
m

print("LDFs:")
l_avg1

print("standard methods:")
Methods


[1] "historical triangle:"


,12,24,36,48,60,72,84,96
2015,276000,412000,463000,435000,440000,445000,468000,470000
2016,319000,513000,570000,858000,859000,899000,924000,NA
2017,538000,702000,737000,765000,950000,1006000,NA,NA
2018,256000,383000,410000,396000,400000,NA,NA,NA
2019,407000,806000,1016000,1020000,NA,NA,NA,NA
2020,422000,643000,772000,NA,NA,NA,NA,NA
2021,350000,631000,NA,NA,NA,NA,NA,NA
2022,350000,NA,NA,NA,NA,NA,NA,NA


[1] "LDFs:"


,24,36,48,60,72,84,96
2015,1.492754,1.123786,0.9395248,1.011494,1.011364,1.051685,1.004274
2016,1.608150,1.111111,1.5052632,1.001166,1.046566,1.027809,NA
2017,1.304833,1.049858,1.0379919,1.241830,1.058947,NA,NA
2018,1.496094,1.070496,0.9658537,1.010101,NA,NA,NA
2019,1.980344,1.260546,1.0039370,NA,NA,NA,NA
2020,1.523697,1.200622,NA,NA,NA,NA,NA
2021,1.802857,NA,NA,NA,NA,NA,NA
2022,NA,NA,NA,NA,NA,NA,NA


[1] "standard methods:"


,24,36,48,60,72,84,96
Avg ex HiLo,1.584710,1.126504,1.0025942,1.010798,1.046566,NA,NA
Wtd 4,1.716376,1.158248,1.1119649,1.079462,NA,NA,NA
Wtd 2,1.650259,1.233954,0.9929874,1.162791,1.053068,1.035714,NA
Wtd All,1.592679,1.147152,1.0869837,1.079462,1.044909,1.035714,1.004274


In [ ]:
##make selects so that they can easily be read in the notebook
##shiny or dataeditR would be better probably

#later periods with limited data
SelNotes[1,"96"]="Not much data; putting some weight on 72-84. This will affect the tail factor select also."
Sel[1,"96"]=(3/4)*Methods["Wtd All","96"]+(1/4)*Methods["Wtd All","84"]

SelNotes[1,"84"]="Not much data, but the decay in Wtd(All) from 60-84 seems reasonable. Decent volatility at those maturities."
Sel[1,"84"]=Methods["Wtd All","84"]

SelNotes[1,"72"]=SelNotes[1,"84"]
Sel[1,"72"]=Methods["Wtd All","72"]

#intermediate periods with volatility
SelNotes[1,"60"]="Relying partially on avg(ex hilo) due to volatility."
Sel[1,"60"]=(1/4)*Methods["Avg ex HiLo","60"]+(3/4)*Methods["Wtd 4","60"]

SelNotes[1,"48"]=SelNotes[1,"60"]
Sel[1,"48"]=(1/4)*Methods["Avg ex HiLo","48"]+(3/4)*Methods["Wtd 4","48"]

#recent periods with maybe trend
SelNotes[1,"36"]="Latest 2 diagonals are above prior history. Putting more weight on recent points."
Sel[1,"36"]=(1/2)*Methods["Wtd 2","36"]+(1/2)*Methods["Wtd 4","36"]

SelNotes[1,"24"]=SelNotes[1,"36"]
Sel[1,"24"]=(1/2)*Methods["Wtd 2","24"]+(1/2)*Methods["Wtd 4","24"]

#tail factor
SelNotes[1,"tail"]="I don't think there is enough data in the tail for me to try to fit anything. Our one value for 84-96 is well below younger values. I'm going to judgmentally assume 3 more periods of development at the same level as 84-96."
Sel[1,"tail"]=Sel[1,"96"]^3

#create cumulative LDFs from incremental
rollprod<- function(x) rollapply(as.numeric(x), width=ncol(Sel), FUN=prod, partial=TRUE, align="left")
Sel["Sel Cum",] <- apply(Sel["Sel Inc",,drop=FALSE],1,rollprod)

#develop to ultimate
Current["ProjUlt",] <- Current["Current",] * Sel["Sel Cum",][1:(ncol(Sel))]

#combine tables for output to excel
m_expanded = cbind(m,t(Current[c("Current","ProjUlt"),ncol(Current):1]))
Methods_expanded = rbind(cbind(Methods,matrix(as.numeric(NA),nrow(Methods),1)),Sel)

#write to excel
newsheet_destructive("triangles")

popsheet("triangles",m_expanded,"Historical Loss and Projected Ultimate",dec0,1,1)
popsheet("triangles",l_avg1,"Age-to-Age Factors",dec3,bounds("triangles")[1]+2,1)
popsheet("triangles",Methods_expanded,"Methods",dec3,bounds("triangles")[1]+2,1)

autowidth("triangles",,12)
wb_save(wb, "output/PYRQ_output.xlsx", overwrite = TRUE)


In [11]:
##create a more generic notes tab
Notes <- data.frame(note=character())
colnames(Notes) <- c("Note")
Notes <- rbind(Notes,data.frame(note="I'm relying on the chain ladder method for this review."))
Notes <- rbind(Notes,data.frame(note="I'm not sure what the basis is, whether it's paid or reported, etc."))
Notes <- rbind(Notes,data.frame(note="But I'm relying on it because it's the only data that I have."))
Notes <- rbind(Notes,data.frame(note="It would be difficult to use an a priori, BF-type, or additive-type method since we don't have exposure."))
Notes <- rbind(Notes,data.frame(note="I think we should request some count and/or exposure data."))
Notes <- rbind(Notes,data.frame(note="Are there good benchmarks available?"))
Notes <- rbind(Notes,data.frame(note="Do we have a sense of what a reasonable tail factor selection would be? I applied a judgmental selection for 96-ultimate."))

newsheet_destructive("notes")
popsheet("notes",Notes,"Notes",dec0,1,1)
autowidth("notes",,12)
wb_save(wb, "output/PYRQ_output.xlsx", overwrite = TRUE)

In [23]:
##output in the notebook also

print("input path:")
input_path

print("output path:")
output_path

print("notes:")
Notes

print("diagnostics:")
test_errors

print("projections:")
m_expanded

print("LDFs")
l_avg1

print("selections:")
Methods_expanded

[1] "input path:"


[1] "sourcedata/PYRQ.xlsx"

[1] "output path:"


[1] "output/PYRQ_output.xlsx"

[1] "notes:"


...

,12,24,36,48,60,72,84,96,Current,ProjUlt
2015,276000,412000,463000,435000,440000,445000,468000,470000,470000,487316.9
2016,319000,513000,570000,858000,859000,899000,924000,NA,924000,969669.0
2017,538000,702000,737000,765000,950000,1006000,NA,NA,1006000,1093426.2
2018,256000,383000,410000,396000,400000,NA,NA,NA,400000,454286.6
2019,407000,806000,1016000,1020000,NA,NA,NA,NA,1020000,1230596.4
2020,422000,643000,772000,NA,NA,NA,NA,NA,772000,1010209.0
2021,350000,631000,NA,NA,NA,NA,NA,NA,631000,987623.0
2022,350000,NA,NA,NA,NA,NA,NA,NA,350000,922138.1


[1] "LDFs"


,24,36,48,60,72,84,96
2015,1.492754,1.123786,0.9395248,1.011494,1.011364,1.051685,1.004274
2016,1.608150,1.111111,1.5052632,1.001166,1.046566,1.027809,NA
2017,1.304833,1.049858,1.0379919,1.241830,1.058947,NA,NA
2018,1.496094,1.070496,0.9658537,1.010101,NA,NA,NA
2019,1.980344,1.260546,1.0039370,NA,NA,NA,NA
2020,1.523697,1.200622,NA,NA,NA,NA,NA
2021,1.802857,NA,NA,NA,NA,NA,NA
2022,NA,NA,NA,NA,NA,NA,NA


[1] "selections:"


,24,36,48,60,72,84,96,
Avg ex HiLo,1.584710,1.126504,1.0025942,1.010798,1.046566,NA,NA,NA
Wtd 4,1.716376,1.158248,1.1119649,1.079462,NA,NA,NA,NA
Wtd 2,1.650259,1.233954,0.9929874,1.162791,1.053068,1.035714,NA,NA
Wtd All,1.592679,1.147152,1.0869837,1.079462,1.044909,1.035714,1.004274,NA
Sel Inc,1.683318,1.196101,1.0846222,1.062296,1.044909,1.035714,1.012134,1.036845
Sel Cum,2.634680,1.565171,1.3085609,1.206467,1.135716,1.086905,1.049425,1.036845
